# Lesson 03 - Padrões de Design Agentes

Nesta lição, exploramos três padrões de design fundamentais para construir agentes de IA eficazes:

1. **Instruções Claras para o Agente** — Criar prompts precisos que definem o papel para guiar o comportamento do agente
2. **Saída Estruturada com Modelos Pydantic** — Garantir que os agentes retornem dados previsíveis e validados
3. **Agentes com Responsabilidade Única** — Projetar agentes focados que fazem bem uma única tarefa

Vamos aplicar cada padrão a um cenário de **recomendador de destinos de viagem**, construindo progressivamente um sistema que pode sugerir destinos, verificar disponibilidade e tratar da logística.


## Configuração


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Padrão 1: Instruções Claras para o Agente

O padrão mais impactante é também o mais simples: escrever instruções claras e detalhadas para o seu agente.

Boas instruções definem:
- **Quem** é o agente (persona e tom)
- **O que** deve fazer (responsabilidades passo a passo)
- **Como** deve comportar-se (restrições e estilo)

Abaixo, criamos um agente concierge de viagens com instruções explícitas que moldam cada resposta que ele produz.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Padrão 2: Saída Estruturada com Modelos Pydantic

O texto livre é útil para conversação, mas os sistemas subsequentes precisam de dados estruturados.
Ao emparelhar **modelos Pydantic** com uma **função de ferramenta**, podemos:

- Definir um esquema exato para a saída do agente
- Validar respostas automaticamente
- Integrar os resultados do agente na lógica da aplicação de forma fiável

Também introduzimos uma ferramenta que devolve detalhes do destino para que o agente baseie as suas recomendações em dados reais.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Padrão 3: Agentes de Responsabilidade Única

Tarefas complexas beneficiam-se de dividir o trabalho entre vários agentes focados, cada um com uma única responsabilidade:

- Um **Especialista em Destinos** que conhece os lugares e a disponibilidade
- Um **Planeador de Logística** que trata de voos, hotéis e itinerários

Isto espelha o princípio da engenharia de software de *separação de responsabilidades* — cada agente é mais fácil de testar, manter e melhorar de forma independente.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Resumo

Nesta lição aplicámos três padrões de design agente a um cenário de recomendação de viagens:

| Padrão | Ideia Principal | Benefício |
|---|---|---|
| **Instruções Claras** | Definir persona, responsabilidades e restrições desde o início | Comportamento do agente consistente e alinhado com a marca |
| **Saída Estruturada** | Usar modelos Pydantic como formato de resposta | Resultados validados e legíveis por máquina |
| **Responsabilidade Única** | Dar a cada agente uma tarefa focada | Mais fácil de testar, manter e compor |

Estes padrões combinam-se naturalmente — pode juntar instruções claras com saída estruturada dentro de um agente de responsabilidade única para construir sistemas robustos e prontos para produção.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido utilizando o serviço de tradução automática [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, esteja ciente de que traduções automáticas podem conter erros ou imprecisões. O documento original na sua língua nativa deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas resultantes da utilização desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
